# Olist E-Commerce — Exploratory Data Analysis

This notebook performs exploratory data analysis (EDA) on top of the SQL-first
analytics already built in `sql/`. Per the project's tech-stack rules, Python
here **supports** SQL rather than replacing it — every dataframe in this
notebook is pulled via SQL queries (either reused directly from `sql/` or close
variants), not recomputed with pandas logic from raw tables.

**Goals of this notebook:**
1. Connect to the live `olist_ecommerce` PostgreSQL database
2. Pull and sanity-check the core datasets behind the Phase 6 SQL findings
3. Profile data quality issues already identified in Phase 2/3 (and confirm they're still true)
4. Set up clean, reusable dataframes for the visualization notebook (`visualization_analysis.ipynb`)


In [3]:
%pip install pandas numpy sqlalchemy
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

print("Libraries loaded.")


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.9 MB 4.1 MB/s eta 0:00:03
   ------- -------------------------------- 1.8/9.9 MB 4.6 MB/s eta 0:00:02
   ----------- ---------------------------- 2.9/9.9 MB 4.7 MB/s eta 0:00:02
   --------------- ------------------------ 3.9/9.9 MB 4.8 MB/s eta 0:00:02
   -------------------- ------------------- 5.0/9.9 MB 4.7 MB/s eta 0:00:02
   ----------------------- ---------------- 5.8/9.9 MB 4.6 MB/s eta 0:00:01
   -------------------------- ------------- 6.6/9.9 MB 4.6 MB/s eta 0:00:01
   ----------------------------- ---------- 7.3/9.9 MB 4.5 MB/s eta 0:00:01
   --------------------------------- ------ 8.4/9.9 MB 4.5 MB/s eta 0:00:01
   -------------------------------------- - 9.4/9.9 MB 4.5 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 4.3 MB/s  0:00:02
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   --------- ----------------

## 1. Database Connection

Connecting via SQLAlchemy + psycopg2 to the `olist_ecommerce` PostgreSQL
database built in Phase 3. Update the connection string below if running
this notebook outside the original setup (e.g., different host/port/credentials).

In [5]:
%pip install psycopg2-binary

   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.8 MB ? eta -:--:--
   ----------- ---------------------------- 0.8/2.8 MB 3.1 MB/s eta 0:00:01
   ------------------------- -------------- 1.8/2.8 MB 3.7 MB/s eta 0:00:01
   ------------------------------------ --- 2.6/2.8 MB 3.5 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 3.5 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# Connection details -- adjust if running locally with different credentials.
# Uses the 'analyst' role created in Phase 3 (sql/schema/01_create_tables.sql
# setup). Local TCP connections require password auth (scram-sha-256) rather
# than the peer/socket auth used for 'postgres' connections in the setup scripts.
DB_USER = "analyst"
DB_PASSWORD = "analyst_pw"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "olist_ecommerce"

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

# Quick connectivity check
test = pd.read_sql("SELECT COUNT(*) AS total_orders FROM orders;", engine)
print("Connected. Total orders:", test['total_orders'][0])


Connected. Total orders: 99441


## 2. Data Quality Recap (validating Phase 2/3 findings against the live DB)

Before any visualization, we re-confirm the key data quality issues identified
earlier in the project. This is a deliberate sanity check, not redundant work —
notebooks should never assume a prior phase's findings are still true without
re-verifying against the live data they're about to visualize.

In [7]:
# customer_id vs customer_unique_id (Phase 2/3 finding)
customer_check = pd.read_sql('''
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT customer_id) AS distinct_customer_id,
           COUNT(DISTINCT customer_unique_id) AS distinct_unique_person
    FROM customers;
''', engine)
display(customer_check)

# order_items grain (multiple items per order)
items_check = pd.read_sql('''
    SELECT COUNT(*) AS total_item_rows,
           COUNT(DISTINCT order_id) AS distinct_orders_with_items
    FROM order_items;
''', engine)
display(items_check)

print(f"\nConfirmed: {customer_check['distinct_customer_id'][0]:,} customer_id values map to only "
      f"{customer_check['distinct_unique_person'][0]:,} unique people.")
print(f"Confirmed: {items_check['total_item_rows'][0]:,} item rows across only "
      f"{items_check['distinct_orders_with_items'][0]:,} distinct orders (multi-item orders exist).")


,total_rows,distinct_customer_id,distinct_unique_person
0,99441,99441,96096


,total_item_rows,distinct_orders_with_items
0,112650,98666



Confirmed: 99,441 customer_id values map to only 96,096 unique people.
Confirmed: 112,650 item rows across only 98,666 distinct orders (multi-item orders exist).


## 3. Core Dataset Pulls

These dataframes mirror the validated logic from `sql/` — same revenue
definition (`order_items.price`, excluding `canceled`/`unavailable` orders),
same `customer_unique_id` discipline for any customer-level analysis.

In [8]:
# Mirrors sql/revenue_analysis/01_monthly_revenue_trend.sql
monthly_revenue = pd.read_sql('''
    SELECT
        DATE_TRUNC('month', o.order_purchase_timestamp)::DATE AS revenue_month,
        ROUND(SUM(oi.price), 2) AS product_revenue,
        COUNT(DISTINCT o.order_id) AS order_count
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    WHERE o.order_status NOT IN ('canceled', 'unavailable')
    GROUP BY DATE_TRUNC('month', o.order_purchase_timestamp)
    ORDER BY revenue_month;
''', engine)

monthly_revenue['revenue_month'] = pd.to_datetime(monthly_revenue['revenue_month'])
print(monthly_revenue.shape)
monthly_revenue.head()


(24, 3)


,revenue_month,product_revenue,order_count
0,2016-09-01,207.86,2
1,2016-10-01,44507.30,290
2,2016-12-01,10.90,1
3,2017-01-01,120098.27,787
4,2017-02-01,244959.35,1718


**Data artifact note (carried over from Query 1's SQL analysis):** the last
month in this series (September 2018) is a known dataset-truncation artifact,
not a real business collapse — the export simply stops mid-month. We flag and
exclude it explicitly below rather than letting it silently distort any
trend visualization.

In [9]:
# Flag the truncated final month explicitly so charts can exclude it
truncation_cutoff = pd.Timestamp('2018-09-01')
monthly_revenue['is_truncated_month'] = monthly_revenue['revenue_month'] >= truncation_cutoff

print("Rows flagged as truncated/incomplete:")
display(monthly_revenue[monthly_revenue['is_truncated_month']])

# Clean version for trend visualizations
monthly_revenue_clean = monthly_revenue[~monthly_revenue['is_truncated_month']].copy()
print(f"\nClean series: {len(monthly_revenue_clean)} months (excludes Sep 2018 truncation artifact)")


Rows flagged as truncated/incomplete:


,revenue_month,product_revenue,order_count,is_truncated_month
23,2018-09-01,145.0,1,True



Clean series: 23 months (excludes Sep 2018 truncation artifact)


## 4. Category Revenue (mirrors Query 5/6/19)

In [10]:
category_revenue = pd.read_sql('''
    SELECT
        COALESCE(ct.product_category_name_english, 'uncategorized') AS category_name,
        ROUND(SUM(oi.price), 2) AS category_revenue,
        COUNT(DISTINCT oi.order_id) AS order_count
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    JOIN products p ON p.product_id = oi.product_id
    LEFT JOIN category_translation ct ON ct.product_category_name = p.product_category_name
    WHERE o.order_status NOT IN ('canceled', 'unavailable')
    GROUP BY category_name
    ORDER BY category_revenue DESC;
''', engine)

category_revenue['pct_of_total'] = (100 * category_revenue['category_revenue']
                                     / category_revenue['category_revenue'].sum()).round(2)
category_revenue['cumulative_pct'] = category_revenue['pct_of_total'].cumsum().round(2)

print(category_revenue.shape)
category_revenue.head(10)


(72, 5)


,category_name,category_revenue,order_count,pct_of_total,cumulative_pct
0,health_beauty,1255695.13,8800,9.31,9.31
1,watches_gifts,1198185.21,5604,8.88,18.19
2,bed_bath_table,1035964.06,9399,7.68,25.87
3,sports_leisure,979740.92,7673,7.26,33.13
4,computers_accessories,904322.02,6654,6.70,39.83
5,furniture_decor,727465.05,6425,5.39,45.22
6,housewares,626825.80,5847,4.65,49.87
7,cool_stuff,620770.49,3616,4.60,54.47
8,auto,586585.73,3872,4.35,58.82
9,garden_tools,481009.94,3505,3.56,62.38


## 5. Customer-Level RFM Data (mirrors Query 8)

Pulling the full customer population (not just the top-20 LIMIT used in the
SQL file for readability) since the visualization notebook will need the
full distribution for the RFM heatmap.

In [11]:
rfm_data = pd.read_sql('''
    WITH order_revenue AS (
        SELECT oi.order_id, ROUND(SUM(oi.price), 2) AS order_revenue
        FROM order_items oi
        GROUP BY oi.order_id
    ),
    dataset_anchor_date AS (
        SELECT MAX(order_purchase_timestamp)::DATE + 1 AS anchor_date FROM orders
    ),
    customer_rfm_raw AS (
        SELECT
            c.customer_unique_id,
            (SELECT anchor_date FROM dataset_anchor_date) - MAX(o.order_purchase_timestamp)::DATE AS recency_days,
            COUNT(DISTINCT o.order_id) AS frequency_orders,
            ROUND(SUM(orev.order_revenue), 2) AS monetary_value
        FROM customers c
        JOIN orders o ON o.customer_id = c.customer_id
        JOIN order_revenue orev ON orev.order_id = o.order_id
        WHERE o.order_status NOT IN ('canceled', 'unavailable')
        GROUP BY c.customer_unique_id
    )
    SELECT
        customer_unique_id, recency_days, frequency_orders, monetary_value,
        6 - NTILE(5) OVER (ORDER BY recency_days ASC) AS recency_score,
        CASE WHEN frequency_orders = 1 THEN 1 WHEN frequency_orders = 2 THEN 3
             WHEN frequency_orders BETWEEN 3 AND 4 THEN 4 ELSE 5 END AS frequency_score,
        NTILE(5) OVER (ORDER BY monetary_value ASC) AS monetary_score
    FROM customer_rfm_raw;
''', engine)

print(rfm_data.shape)
rfm_data.describe()


(94983, 7)


,recency_days,frequency_orders,monetary_value,recency_score,frequency_score,monetary_score
count,94983.000000,94983.000000,94983.000000,94983.000000,94983.000000,94983.000000
mean,288.334197,1.033859,142.071747,3.000032,1.063475,2.999968
std,152.984601,0.210811,216.074999,1.414217,0.362458,1.414217
min,45.000000,1.000000,0.850000,1.000000,1.000000,1.000000
25%,164.000000,1.000000,47.900000,2.000000,1.000000,2.000000
50%,269.000000,1.000000,89.890000,3.000000,1.000000,3.000000
75%,397.000000,1.000000,155.000000,4.000000,1.000000,4.000000
max,774.000000,16.000000,13440.000000,5.000000,5.000000,5.000000


## 6. Cohort Retention Data (mirrors Query 10)

In [12]:
cohort_retention = pd.read_sql('''
    WITH customer_first_order AS (
        SELECT c.customer_unique_id,
               MIN(DATE_TRUNC('month', o.order_purchase_timestamp))::DATE AS cohort_month
        FROM customers c JOIN orders o ON o.customer_id = c.customer_id
        WHERE o.order_status NOT IN ('canceled', 'unavailable')
        GROUP BY c.customer_unique_id
    ),
    customer_order_months AS (
        SELECT DISTINCT c.customer_unique_id,
               DATE_TRUNC('month', o.order_purchase_timestamp)::DATE AS order_month
        FROM customers c JOIN orders o ON o.customer_id = c.customer_id
        WHERE o.order_status NOT IN ('canceled', 'unavailable')
    ),
    cohort_activity AS (
        SELECT cfo.cohort_month, com.order_month,
               (DATE_PART('year', com.order_month) - DATE_PART('year', cfo.cohort_month)) * 12
                   + (DATE_PART('month', com.order_month) - DATE_PART('month', cfo.cohort_month)) AS months_since_cohort,
               com.customer_unique_id
        FROM customer_first_order cfo
        JOIN customer_order_months com ON com.customer_unique_id = cfo.customer_unique_id
        WHERE cfo.cohort_month >= '2017-01-01'
    ),
    cohort_sizes AS (
        SELECT cohort_month, COUNT(*) AS cohort_size
        FROM customer_first_order
        WHERE cohort_month >= '2017-01-01'
        GROUP BY cohort_month
    )
    SELECT ca.cohort_month, cs.cohort_size, ca.months_since_cohort,
           COUNT(DISTINCT ca.customer_unique_id) AS active_customers,
           ROUND(100.0 * COUNT(DISTINCT ca.customer_unique_id) / cs.cohort_size, 2) AS retention_pct
    FROM cohort_activity ca
    JOIN cohort_sizes cs ON cs.cohort_month = ca.cohort_month
    WHERE ca.months_since_cohort BETWEEN 0 AND 6
    GROUP BY ca.cohort_month, cs.cohort_size, ca.months_since_cohort
    ORDER BY ca.cohort_month, ca.months_since_cohort;
''', engine)

cohort_retention['cohort_month'] = pd.to_datetime(cohort_retention['cohort_month'])
print(cohort_retention.shape)
cohort_retention.head(10)


(120, 5)


,cohort_month,cohort_size,months_since_cohort,active_customers,retention_pct
0,2017-01-01,752,0.0,752,100.00
1,2017-01-01,752,1.0,3,0.40
2,2017-01-01,752,2.0,2,0.27
3,2017-01-01,752,3.0,1,0.13
4,2017-01-01,752,4.0,3,0.40
5,2017-01-01,752,5.0,1,0.13
6,2017-01-01,752,6.0,3,0.40
7,2017-02-01,1690,0.0,1690,100.00
8,2017-02-01,1690,1.0,4,0.24
9,2017-02-01,1690,2.0,5,0.30


## 7. Delivery Delay & Review Score Data (mirrors Query 14/15)

In [13]:
delivery_reviews = pd.read_sql('''
    WITH order_review_avg AS (
        SELECT order_id, AVG(review_score) AS avg_order_review_score
        FROM order_reviews GROUP BY order_id
    )
    SELECT
        o.order_id,
        EXTRACT(DAY FROM (o.order_delivered_customer_date - o.order_estimated_delivery_date))::INT AS delay_days,
        ora.avg_order_review_score
    FROM orders o
    JOIN order_review_avg ora ON ora.order_id = o.order_id
    WHERE o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL;
''', engine)

print(delivery_reviews.shape)
delivery_reviews.describe()


(95830, 3)


,delay_days,avg_order_review_score
count,95830.000000,95830.000000
mean,-10.994021,4.156023
std,9.952409,1.283745
min,-146.000000,1.000000
25%,-16.000000,4.000000
50%,-11.000000,5.000000
75%,-6.000000,5.000000
max,188.000000,5.000000


## 8. Seller Performance Data (mirrors Query 13)

In [14]:
seller_performance = pd.read_sql('''
    WITH order_review_avg AS (
        SELECT order_id, AVG(review_score) AS avg_order_review_score
        FROM order_reviews GROUP BY order_id
    ),
    seller_orders AS (
        SELECT oi.seller_id, oi.order_id, o.order_delivered_customer_date,
               o.order_estimated_delivery_date, ora.avg_order_review_score
        FROM order_items oi
        JOIN orders o ON o.order_id = oi.order_id
        LEFT JOIN order_review_avg ora ON ora.order_id = oi.order_id
        WHERE o.order_status NOT IN ('canceled', 'unavailable')
    )
    SELECT
        seller_id,
        COUNT(DISTINCT order_id) AS order_count,
        ROUND(AVG(avg_order_review_score), 2) AS avg_review_score,
        ROUND(AVG(EXTRACT(DAY FROM (order_delivered_customer_date - order_estimated_delivery_date)))
              FILTER (WHERE order_delivered_customer_date IS NOT NULL
                        AND order_estimated_delivery_date IS NOT NULL), 2) AS avg_delivery_delay_days
    FROM seller_orders
    GROUP BY seller_id
    HAVING COUNT(DISTINCT order_id) >= 20;
''', engine)

print(seller_performance.shape)
seller_performance.describe()


(814, 4)


,order_count,avg_review_score,avg_delivery_delay_days
count,814.000000,814.000000,814.000000
mean,107.366093,4.089963,-11.330061
std,184.777686,0.364551,3.217499
min,20.000000,1.260000,-41.910000
25%,31.000000,3.900000,-13.110000
50%,52.000000,4.130000,-10.965000
75%,104.000000,4.330000,-9.422500
max,1847.000000,5.000000,1.410000


## 9. Geographic Revenue Data (mirrors Query 16/20)

In [15]:
state_metrics = pd.read_sql('''
    WITH order_revenue AS (
        SELECT oi.order_id, ROUND(SUM(oi.price), 2) AS order_revenue
        FROM order_items oi GROUP BY oi.order_id
    )
    SELECT
        c.customer_state,
        COUNT(DISTINCT o.order_id) AS order_count,
        ROUND(SUM(orev.order_revenue), 2) AS state_revenue,
        ROUND(SUM(orev.order_revenue) / COUNT(DISTINCT o.order_id), 2) AS avg_order_value
    FROM customers c
    JOIN orders o ON o.customer_id = c.customer_id
    JOIN order_revenue orev ON orev.order_id = o.order_id
    WHERE o.order_status NOT IN ('canceled', 'unavailable')
    GROUP BY c.customer_state
    ORDER BY state_revenue DESC;
''', engine)

print(state_metrics.shape)
correlation = state_metrics['order_count'].corr(state_metrics['avg_order_value'])
print(f"\nCorrelation between order_count and avg_order_value across states: {correlation:.3f}")
state_metrics.head(10)


(27, 4)

Correlation between order_count and avg_order_value across states: -0.526


,customer_state,order_count,state_revenue,avg_order_value
0,SP,41125,5163867.22,125.57
1,RJ,12697,1811623.42,142.68
2,MG,11496,1573508.20,136.87
3,RS,5415,742559.78,137.13
4,PR,4982,676883.06,135.87
5,SC,3599,518578.28,144.09
6,BA,3344,507108.83,151.65
7,DF,2120,300886.45,141.93
8,GO,1998,287870.46,144.08
9,ES,2018,273532.13,135.55


## 10. Save Cleaned Dataframes for the Visualization Notebook

Persisting these as CSVs in a local `_cache` folder so
`visualization_analysis.ipynb` doesn't need to re-run every SQL query --
it loads directly from here. This also makes the visualization notebook
runnable independently once this EDA notebook has been run once.

In [17]:
import os
os.makedirs('_cache', exist_ok=True)

monthly_revenue.to_csv('_cache/monthly_revenue.csv', index=False)
category_revenue.to_csv('_cache/category_revenue.csv', index=False)
rfm_data.to_csv('_cache/rfm_data.csv', index=False)
cohort_retention.to_csv('_cache/cohort_retention.csv', index=False)
delivery_reviews.to_csv('_cache/delivery_reviews.csv', index=False)
seller_performance.to_csv('_cache/seller_performance.csv', index=False)
state_metrics.to_csv('_cache/state_metrics.csv', index=False)

print("All cleaned dataframes cached to notebooks/_cache/ for use in visualization_analysis.ipynb")


All cleaned dataframes cached to notebooks/_cache/ for use in visualization_analysis.ipynb


## Summary

This EDA notebook confirmed (against the live database, not just assumed from
earlier phases):
- The `customer_id` vs `customer_unique_id` distinction is real and necessary
- `order_items` is at line-item grain, not order grain
- September 2018 is a genuine data-truncation artifact, now explicitly flagged
- Core revenue, customer, cohort, delivery, seller, and geographic datasets
  are pulled, validated in shape, and cached for the next notebook

**Next:** `visualization_analysis.ipynb` builds the insight-focused charts on
top of these cached dataframes.